# 35 — EDA: ICWSM & JCDL Award Data

Exploratory analysis of `icwsm_jcdl_awards_raw.csv` produced by notebook `01b`.  

Sections:
1. Load & sanity check  
2. Coverage — records per year × conference  
3. OpenAlex match rate by year and award type  
4. Citation distribution (matched papers only)  
5. Top-10 cited papers per award type  
6. Missing data audit — unmatched papers table  


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pd.set_option('display.max_colwidth', 90)

df = pd.read_csv('../data/raw/icwsm_jcdl_awards_raw.csv')
df['matched'] = df['openalex_id'].notna() & (df['openalex_id'] != '')
df['cited_by_count'] = pd.to_numeric(df['cited_by_count'], errors='coerce')

print(f'Total records : {len(df)}')
print(f'Matched to OA : {df.matched.sum()} ({df.matched.mean():.1%})')
print(f'Year range    : {df.year.min()} – {df.year.max()}')
print()
df.dtypes

## 1. Award type breakdown

In [ ]:
breakdown = (
    df.groupby(['conference', 'award_type'])
      .agg(count=('paper_title', 'count'),
           matched=('matched', 'sum'),
           median_cites=('cited_by_count', 'median'))
      .reset_index()
)
breakdown['match_rate'] = (breakdown['matched'] / breakdown['count']).map('{:.0%}'.format)
breakdown

## 2. Coverage — records per year × conference

In [ ]:
pivot = df.groupby(['year', 'conference']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
pivot.plot(kind='bar', ax=ax, width=0.75, colormap='tab10')
ax.set_xlabel('Year')
ax.set_ylabel('Awards')
ax.set_title('Award records per year by conference')
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(title='Conference')
plt.tight_layout()
plt.savefig('../data/raw/eda_coverage_per_year.png', dpi=150)
plt.show()

## 3. OpenAlex match rate by year

In [ ]:
match_by_year = (
    df.groupby(['year', 'conference'])['matched']
      .agg(['sum', 'count'])
      .reset_index()
)
match_by_year['rate'] = match_by_year['sum'] / match_by_year['count']

fig, ax = plt.subplots(figsize=(12, 4))
for conf, grp in match_by_year.groupby('conference'):
    ax.plot(grp['year'], grp['rate'], marker='o', label=conf)
ax.axhline(0.85, color='grey', linestyle='--', linewidth=0.8, label='85% threshold')
ax.set_xlabel('Year')
ax.set_ylabel('Match rate')
ax.set_title('OpenAlex match rate by year & conference')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.savefig('../data/raw/eda_match_rate_by_year.png', dpi=150)
plt.show()

## 4. Citation distribution (matched papers)

In [ ]:
matched = df[df['matched'] & df['cited_by_count'].notna()].copy()

# Summary stats
stats = (
    matched.groupby(['conference', 'award_type'])['cited_by_count']
           .describe(percentiles=[.25, .5, .75])
           .round(1)
)
print(stats.to_string())

# Box plots per award_type, coloured by conference
award_types = matched['award_type'].unique()
fig, ax = plt.subplots(figsize=(13, 5))

labels, data_sets = [], []
for conf in ['ICWSM', 'JCDL']:
    sub = matched[matched['conference'] == conf]
    for atype in sorted(sub['award_type'].unique()):
        vals = sub[sub['award_type'] == atype]['cited_by_count'].dropna().values
        if len(vals) > 0:
            data_sets.append(vals)
            labels.append(f'{conf}\n{atype}')

ax.boxplot(data_sets, labels=labels, vert=True, patch_artist=True,
           medianprops=dict(color='black', linewidth=1.5))
ax.set_ylabel('cited_by_count')
ax.set_title('Citation distribution by conference × award type')
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../data/raw/eda_citation_boxplot.png', dpi=150)
plt.show()

## 5. Top-10 cited papers per award type

In [ ]:
FOCUS_AWARDS = ['Vannevar Bush Best Paper', 'Best Paper', 'Test of Time', 'Best Student Paper']

for atype in FOCUS_AWARDS:
    sub = matched[matched['award_type'] == atype].nlargest(10, 'cited_by_count')
    if sub.empty:
        continue
    print(f"\n=== {atype} — Top 10 by citations ==={chr(10)}")
    display(sub[['year', 'conference', 'paper_title', 'cited_by_count']].reset_index(drop=True))

## 6. Missing data audit — unmatched papers

In [ ]:
unmatched = df[~df['matched']].copy()
print(f'Unmatched records: {len(unmatched)}')
print()

# By year — helps spot systematic gaps (e.g. old papers pre-2005)
print('Unmatched by year:')
print(unmatched.groupby('year').size().to_string())
print()

# Full table
display(unmatched[['year', 'conference', 'award_type', 'paper_title', 'authors']].reset_index(drop=True))

## 7. Citation distribution over time (Vannevar Bush + ICWSM Best Paper)

Scatter of `cited_by_count` vs `year` for the two main best-paper awards — useful to see if older papers have had more time to accumulate citations.

In [ ]:
main_awards = matched[matched['award_type'].isin(['Vannevar Bush Best Paper', 'Best Paper'])].copy()

fig, ax = plt.subplots(figsize=(10, 5))
colors = {'ICWSM': '#e15759', 'JCDL': '#4e79a7'}
for conf, grp in main_awards.groupby('conference'):
    ax.scatter(grp['year'], grp['cited_by_count'],
               color=colors.get(conf, 'grey'), alpha=0.7, label=conf, s=40)

ax.set_xlabel('Award year')
ax.set_ylabel('cited_by_count (OpenAlex)')
ax.set_title('Best Paper citations vs award year (ICWSM & JCDL Vannevar Bush)')
ax.legend()
plt.tight_layout()
plt.savefig('../data/raw/eda_bestpaper_citations_over_time.png', dpi=150)
plt.show()